# Schema Drift Detection for Mirrored Cosmos DB Tables

This notebook profiles mirrored Cosmos DB tables in OneLake to detect schema drift and columns that may be suffering from silent type coercion to `NULL`.

In [ ]:
from pyspark.sql import functions as F

SOURCE_DELTA_PATH = "abfss://<workspace>@onelake.dfs.fabric.microsoft.com/<lakehouse>/Tables/cosmos_mirror_sales"
NULL_THRESHOLD = 0.30
REPORT_TABLE = "monitoring.cosmos_schema_drift_report"
WRITE_REPORT = True

mirror_df = spark.read.format("delta").load(SOURCE_DELTA_PATH)
mirror_df.printSchema()

In [ ]:
row_count = mirror_df.count()
null_count_exprs = [
    F.sum(F.when(F.col(c).isNull(), 1).otherwise(0)).alias(c)
    for c in mirror_df.columns
]
null_counts = mirror_df.agg(*null_count_exprs).collect()[0].asDict()
null_pct_df = spark.createDataFrame(
    [(c, int(v), round(float(v) / row_count, 4)) for c, v in null_counts.items()],
    ["column_name", "null_count", "null_pct"]
)
display(null_pct_df.orderBy(F.col("null_pct").desc()))

In [ ]:
high_null_columns = null_pct_df.filter(F.col("null_pct") >= F.lit(NULL_THRESHOLD)).orderBy(F.col("null_pct").desc())
display(high_null_columns)

In [ ]:
problem_columns = [row.column_name for row in high_null_columns.collect()]
sample_views = {}
for column_name in problem_columns:
    sample_views[column_name] = {
        "nonnull": mirror_df.filter(F.col(column_name).isNotNull()).limit(5),
        "nulls": mirror_df.filter(F.col(column_name).isNull()).limit(5)
    }

for column_name, samples in sample_views.items():
    print(f"=== Column: {column_name} | non-null samples ===")
    display(samples["nonnull"])
    print(f"=== Column: {column_name} | null samples ===")
    display(samples["nulls"])

In [ ]:
schema_summary = spark.createDataFrame(
    [(field.name, field.dataType.simpleString(), field.nullable) for field in mirror_df.schema.fields],
    ["column_name", "spark_type", "nullable"]
)
issue_report = (
    schema_summary
    .join(null_pct_df, on="column_name", how="left")
    .withColumn(
        "issue_flag",
        F.when(F.col("null_pct") >= F.lit(NULL_THRESHOLD), F.lit("high_null_rate")).otherwise(F.lit("ok"))
    )
    .orderBy(F.col("null_pct").desc_nulls_last())
)
display(issue_report)

In [ ]:
if WRITE_REPORT:
    (
        issue_report.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(REPORT_TABLE)
    )

spark.sql(f"SELECT * FROM {REPORT_TABLE} ORDER BY null_pct DESC NULLS LAST").show(truncate=False) if WRITE_REPORT else None